In [1]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")
groq_api_key



'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'

In [ ]:
from langchain_groq import ChatGroq
model=ChatGroq(model="qwen/qwen3.6-27b",groq_api_key=groq_api_key,)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x00000190B77B0910>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000190B77B1310>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [9]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

C:\Users\utyag\AppData\Local\Temp\ipykernel_9516\2274412368.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
config={"configurable":{"session_id":"chat1"}}

In [11]:
from langchain_core.messages import HumanMessage

response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Ujjwal and I am a Senior AI/ML Engineer")],
    config=config
)

In [15]:
print(response.content.split("</think>")[-1])



Hi Ujjwal! Great to meet you. 🙌

As a Senior AI/ML Engineer, you're likely navigating complex challenges like scalable MLOps, model optimization, production deployment, LLM fine-tuning, or cutting-edge research. I'm here to help with anything from architecture design and debugging training pipelines to evaluating trade-offs, staying current with papers, or brainstorming system-level solutions.

What are you currently working on, or how can I best support you today?


In [16]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)
print(response.content.split("</think>")[-1])




Your name is Ujjwal. How can I help you today?


In [17]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1,
)
print(response.content.split("</think>")[-1])




I don't actually know your name! I don't have memory of past conversations or access to personal information. If you'd like to tell me your name, I'll be happy to use it during our chat. What should I call you?


In [18]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
print(response.content.split("</think>")[-1])



Nice to meet you, John! I'll be sure to call you that from now on. How can I help you today? 😊


In [19]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
print(response.content.split("</think>")[-1])



Your name is John! How can I help you today? 😊


### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [20]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model
chain

ChatPromptTemplate(input_variables=['messages'], input_types={'messages': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')] | typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')] | typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')] | typing.Annotated[langchain_core.messag

In [21]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Ujjwal")]})

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi My name is Ujjwal"\n   - This is a simple greeting and introduction.\n\n2.  **Identify Key Elements:**\n   - Greeting: "Hi"\n   - Name: "Ujjwal"\n   - Intent: Introduce themselves, likely expecting a friendly response and acknowledgment.\n\n3.  **Determine Response Strategy:**\n   - Acknowledge the greeting\n   - Use their name\n   - Be friendly and welcoming\n   - Offer assistance\n   - Keep it concise\n\n4.  **Draft Response (Mental):**\n   Hi Ujjwal! Nice to meet you. How can I assist you today?\n\n5.  **Refine Response:**\n   - Check tone: Friendly, professional\n   - Check accuracy: Uses correct name\n   - Check purpose: Opens door for further conversation\n   - All good.\n\n6.  **Final Output Generation:** (matches the refined draft)\n   "Hi Ujjwal! Nice to meet you. How can I assist you today?"✅\n</think>\n\nHi Ujjwal! Nice to meet you. How can I assist you today? 😊', ad

In [22]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [23]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Krish")],
    config=config
)

response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi My name is Krish"\n   - This is a simple greeting and introduction.\n   - Key information: Name = Krish\n   - Intent: Greeting, establishing context for conversation\n\n2.  **Identify Response Requirements:**\n   - Acknowledge the greeting\n   - Use the user\'s name (Krish)\n   - Be polite and welcoming\n   - Offer assistance\n   - Keep it concise\n\n3.  **Draft Response (Mental):**\n   Hi Krish! Nice to meet you. How can I help you today?\n\n4.  **Refine Response:**\n   - Check tone: Friendly, professional\n   - Check accuracy: Uses correct name\n   - Check completeness: Greeting + offer to help\n   - Matches standard AI response patterns\n\n   "Hi Krish! Nice to meet you. How can I assist you today?"\n\n5.  **Final Output Generation:** (matches the refined version)✅\n</think>\n\nHi Krish! Nice to meet you. How can I help you today? 😊', additional_kwargs={}, response_metadata=

In [24]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

print(response.content.split("</think>")[-1])



Your name is Krish! 😊


In [39]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [40]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")],"language":"Hindi"})
print(response.content.split("</think>")[-1])



नमस्ते कृष! आपका नाम जानकर बहुत अच्छा लगा। मैं आपकी कैसे मदद कर सकता हूँ? यहाँ कुछ भी पूछें या बताएँ, मैं पूरी कोशिश करूँगा। 😊


Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [41]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [43]:
config = {"configurable": {"session_id": "chat5"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Krish")],"language":"Hindi"},
    config=config
)
print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "Hi My name is Krish"
   - Language: English
   - Intent: Greeting and introducing themselves

2.  **Identify Constraints:**
   - System prompt: "Answer all questions to the best of your ability in Hindi."
   - This means I must respond in Hindi, regardless of the input language.

3.  **Formulate Response (Internal Translation & Structuring):**
   - Acknowledge the greeting: "Hi" -> "नमस्ते" or "हैलो"
   - Acknowledge the name: "My name is Krish" -> "आपका नाम कृष है"
   - Respond politely in Hindi: "नमस्ते कृष! आपका नाम जानकर अच्छा लगा। मैं आपकी कैसे मदद कर सकता हूँ?"
   - Check tone: Friendly, helpful, natural Hindi
   - Ensure compliance: Entire response in Hindi

4.  **Refine Response (Hindi):**
   नमस्ते कृष! आपका नाम जानकर बहुत अच्छा लगा। मैं आपकी कैसे मदद कर सकता हूँ? यहाँ कुछ भी पूछें या बताएँ, मैं पूरी कोशिश करूँगा।

5.  **Final Check against Constraints:**
   - All in Hindi? Yes.
   - Addresses th

In [44]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [31]:
response.content

'आपका नाम कृष है। 😊 \n'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [45]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\langchain_core\language_models\base.py:448: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\utyag\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [46]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What ice cream do i like"\n2.  **Identify Key Information Needed:** To answer this, I would need to know the user\'s personal preferences, which I don\'t have access to unless they\'ve told me in this conversation or it\'s stored in memory (which it isn\'t, based on the brief conversation history).\n3.  **Check Conversation History:** The previous messages are:\n   - User: "whats 2 + 2" -> Assistant: "4"\n   - User: "thanks" -> Assistant: "no problem!"\n   - User: "having fun?" -> Assistant: "yes!"\n   - User: "What ice cream do i like"\n   No mention of ice cream preferences.\n4.  **Determine Response Strategy:** \n   - Acknowledge that I don\'t know their preference\n   - Explain that I don\'t have access to personal information unless they tell me\n   - Ask them what their favorite ice cream is so I can remember it for the conversation\n   - Keep it friendly and conversational\n5.  **Draft Response

In [47]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "what math problem did i ask"\n2.  **Review Conversation History:**\n   - User: "whats 2 + 2"\n   - Assistant: "4"\n   - User: "thanks"\n   - Assistant: "no problem!"\n   - User: "having fun?"\n   - Assistant: "yes!"\n   - User: "what math problem did i ask"\n3.  **Identify the Math Problem:** The only math problem asked was "2 + 2".\n4.  **Formulate Response:** State clearly what the math problem was, keeping it concise and in English as instructed.\n   - Draft: "You asked what 2 + 2 equals."\n   - Check against constraints: English? Yes. Accurate? Yes. Concise? Yes.\n5.  **Final Output Generation:** "You asked what 2 + 2 equals." (or similar)✅\n</think>\n\nYou asked what 2 + 2 equals.'

In [48]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [49]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "whats my name?"\n2.  **Check Context/History:** I need to look at the conversation history provided.\n   - User: "I like vanilla ice cream"\n   - Me: "nice"\n   - User: "whats 2 + 2"\n   - Me: "4"\n   - User: "thanks"\n   - Me: "no problem!"\n   - User: "having fun?"\n   - Me: "yes!"\n   - User: "whats my name?"\n3.  **Identify Key Information:** The user has not provided their name anywhere in the conversation.\n4.  **Formulate Response:** I should politely state that I don\'t know their name because they haven\'t told me, and ask if they\'d like to share it. Keep it concise and friendly, matching the tone of the conversation.\n5.  **Draft Response:** "You haven\'t told me your name yet! What should I call you?"\n6.  **Check Against Constraints:** The response is in English, addresses the question directly, and is appropriate. No extra fluff. Matches the conversational style.\n7.  **Final Output Gene

In [50]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "what math problem did i ask?"\n2.  **Identify Key Issue:** This is the first message in the conversation. There is no prior context or history provided.\n3.  **Determine Response Strategy:** \n   - Acknowledge that this is the start of the conversation.\n   - State clearly that no math problem has been asked yet.\n   - Invite the user to share their math problem so I can help.\n4.  **Draft Response (Internal Refinement):** "You haven\'t asked a math problem yet—this is the first message in our conversation! Feel free to share the problem you\'d like help with, and I’ll do my best to solve it for you."\n5.  **Check Against Guidelines:** \n   - Language: English (matches requirement)\n   - Tone: Helpful, clear\n   - Accuracy: Correctly identifies lack of prior context\n   - Concise: Yes\n6.  **Final Output Generation:** (Matches the drafted response)✅\n</think>\n\nYou haven\'t asked a math problem yet—t